# Day 30: 综合验收

组合模型、紧凑 KV、分页分配、分桶合批、Prefix Cache、正确性与计时。口述一次请求完整生命周期，并说明与真实 vLLM 的差距。

参考源码：`learning/mini_vllm/README.md`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 30完整讲义](../docs/DAY30_CAPSTONE.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

回到同一条请求，把所有技术放对位置：

```text
文本 → tokenizer → Request → 准入/调度 → Prefill
                 ↓                      ↓
              停止条件          权重计算、RoPE/GQA、KV写入
                 ↑                      ↓
                 └─ 采样 ← logits ← 多轮Decode
                                          ↓
                              完成/异常 → 请求状态及页回收
```

优化的对象不同：GQA/MLA影响KV表示，FlashAttention影响Attention中间IO，
Paged KV影响分配映射，continuous batching影响运行集合，prefix cache影响重复前缀计算。
它们可以组合，但名字同时出现不代表所有实现和生产能力都已经具备。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(30, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

能区分模型数学、kernel 优化、内存分配、调度和服务协议吗？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：当前项目与生产vLLM差在哪里？**
答：权重加载/模型规模、真实paged kernel、异长合批、抢占/取消、
分布式/量化、协议、安全、监控和服务压测等；要按实测范围逐项回答。

**问：怎样证明“从Cache读到了正确请求”？**
答：请求身份、逻辑/物理映射、有效长度和实际数值对照，不能只看shape。

**问：最佳优化方向是什么？**
答：由当前性能/质量/成本目标和profiling证据决定，不按热门技术列表决定。

**问：材料齐备和个人完成如何区分？**
答：课程提供参考；个人要独立口述、改写、构造失败、复现并在新输入上解释权衡。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY30_CAPSTONE.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。